# Qutrit Post-Processing: Choi Matrix Reconstruction and Fidelity Evaluation

This notebook post-processes the raw qutrit process tomography data collected by
`SimulationQutrit.ipynb` to produce the process fidelities reported in the manuscript.

For each of the 40 metric operators the notebook:
1. Loads the REM-corrected qutrit count data from `data/raw_tomography/qutrit_raw_counts/`.
2. Fixes any residual negative probabilities from REM and renormalises.
3. Reconstructs the Choi matrix via linear inversion.
4. Projects the Choi matrix onto the set of physical (positive semidefinite) operators.
5. Computes the process fidelity against the ideal Choi matrix and compares against
   the adversarial threshold.

Results are saved to `data/processed/experimental_fidelities_master.csv`.

In [ ]:
import sys
import csv
import os
from math import pi

import numpy as np
import pandas as pd
import scipy as sp
from scipy import linalg as la

from forest.benchmarking.tomography import (
    _pauli_process_tomo_settings,
    proj_choi_to_physical,
)
from forest.benchmarking.observable_estimation import *
from forest.benchmarking.operator_tools import superoperator_transformations as supt

if sys.version_info < (3, 7):
    from pyquil.external.dataclasses import dataclass
else:
    from dataclasses import dataclass

## Metric operator parameter grid

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Helper functions

In [ ]:
def get_etahalf(r, theta, phi):
    """Construct the square root of the metric operator eta from its parameters.

    Args:
        r: radial metric parameter.
        theta: polar angle parameter (radians).
        phi: azimuthal angle parameter (radians).

    Returns:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.
    """
    eta_half = np.array([
        [
            np.cos(theta/2)**2 + np.sqrt(r) * np.sin(theta/2)**2,
            (np.cos(phi) - 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r))
        ],
        [
            (np.cos(phi) + 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r)),
            np.sqrt(r) * np.cos(theta/2)**2 + np.sin(theta/2)**2
        ]
    ])
    return eta_half


def get_idealchoi(eta_half):
    """Compute the ideal Choi matrix for the IP-change channel defined by eta_half.

    Args:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.

    Returns:
        choi: (4, 4) complex array, the ideal Choi matrix.
    """
    max_entangled = np.array([[1, 0, 0, 1]]).T / np.sqrt(2)
    identity = np.eye(2)
    A = np.kron(identity, eta_half)
    choi = 2 * A @ max_entangled @ max_entangled.conj().T @ A.conj().T
    return choi


def get_fidelity(ideal_choi, estimated_choi):
    """Compute the process fidelity between the ideal and estimated Choi matrices.

    Uses the quantum fidelity formula F = Tr[sqrt(sqrt(A) B sqrt(A))] / (sqrt(Tr[A]) sqrt(Tr[B])).

    Args:
        ideal_choi: (4, 4) complex array, the ideal Choi matrix.
        estimated_choi: (4, 4) complex array, the experimentally reconstructed Choi matrix.

    Returns:
        fidelity: real scalar process fidelity.
    """
    sqrt_ideal = sp.linalg.sqrtm(ideal_choi)
    numerator = np.trace(sp.linalg.sqrtm(sqrt_ideal @ estimated_choi @ sqrt_ideal))
    denominator = np.sqrt(np.trace(ideal_choi)) * np.sqrt(np.trace(estimated_choi))
    return (numerator / denominator).real

## Build the linear inversion measurement matrix

The measurement matrix is constructed once from the standard Pauli process tomography
settings and is shared across all 40 metrics. It is augmented with identity rows to
fix the trace of the reconstructed Choi matrix.

In [ ]:
qs = [0]
tomo_expt_settings = list(_pauli_process_tomo_settings(qs))

# Input state density matrices for each preparation setting
# (ordered X+, X-, Y+, Y-, Z+, Z-; one per 3-observable block)
input_states = [
    state2matrix(tomo_expt_settings[i * 3].in_state, qs)
    for i in range(6)
]

# Measurement matrix from Pauli observables
measurement_matrix = np.vstack([
    vec(np.kron(
        state2matrix(setting.in_state, qs).conj(),
        pauli2matrix(setting.observable, qs)
    )).conj().T
    for setting in tomo_expt_settings
])

# Augment with identity rows to fix the trace (one per input state)
identity_rows = np.vstack([
    vec(np.kron(rho.conj(), np.eye(2))).conj().T
    for rho in input_states
])
measurement_matrix = np.vstack((measurement_matrix, identity_rows))

## Reconstruct Choi matrices and evaluate fidelities

For each metric, residual negative probabilities from REM are clipped to zero and
the counts renormalised before linear inversion. The reconstructed Choi matrix is
then projected to the nearest physical (positive semidefinite) operator before
computing the process fidelity.

In [ ]:
INPUT_DIR = "data/raw_tomography/qutrit_raw_counts"
OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

grid = eta_grid()
rows = []

for k in range(40):
    # Load REM-corrected counts
    data_path = os.path.join(INPUT_DIR, f"qutrit_tomography_metric_{k+1:02d}.csv")
    data = pd.read_csv(data_path).values.tolist()

    # Clip residual negative probabilities and renormalise
    for idx in range(18):
        rem_counts = [max(0, p) for p in data[idx][3:6]]
        total = sum(rem_counts)
        data[idx][3:6] = [p / total for p in rem_counts]

    # Build expectation value vector:
    # Each tomography setting yields p(|0>) - p(|1>) as the Pauli expectation value,
    # and 1 - p(|2>) as the trace normalisation for each input state.
    expectations = []
    for j in range(18):
        row = pd.to_numeric(pd.Series(data[j])).values
        expectations.append(row[3] - row[4])
    for j in range(0, 18, 3):
        row = pd.to_numeric(pd.Series(data[j])).values
        expectations.append(1 - row[5])
    expectations = np.array(expectations)[:, np.newaxis]

    # Reconstruct Choi matrix via linear inversion
    rho = np.linalg.pinv(measurement_matrix) @ expectations
    choi = unvec(rho)

    # Project to nearest physical (positive semidefinite) operator
    choi_proj = proj_choi_to_physical(choi, False)

    # Compute process fidelity against ideal Choi matrix
    eta_half = get_etahalf(grid[k][0], grid[k][1], grid[k][2])
    ideal_choi = get_idealchoi(eta_half)
    fidelity = get_fidelity(ideal_choi, choi_proj)

    # Adversarial threshold for certifying a genuinely higher-dimensional process
    r = grid[k][0]
    threshold = (1 + np.sqrt(r)) / (np.sqrt(2) * np.sqrt(1 + r))

    rows.append({
        'index': k + 1,
        'r': r,
        'theta_pi': grid[k][1] / pi,
        'phi_pi': grid[k][2] / pi,
        'threshold_fidelity': round(threshold, 3),
        'qutrit_fidelity': round(fidelity, 3),
        'qutrit_passed': fidelity > threshold,
    })

# Save results
out_path = os.path.join(OUTPUT_DIR, "experimental_fidelities_qutrit.csv")
pd.DataFrame(rows).to_csv(out_path, index=False)
print(f"Done. Results saved to {out_path}")
print(f"Metrics passing threshold: {sum(r['qutrit_passed'] for r in rows)}/40")